In [1]:
pip install ncps tensorflow matplotlib seaborn opencv-python tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install keras-ncp

Note: you may need to restart the kernel to use updated packages.


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import PIL
from PIL import Image
from numpy import ndarray
from tqdm import tqdm

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
from dataclasses import dataclass, field, asdict
from typing import Tuple, Dict, Optional, List, Iterable, Union

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import kerasncp as kncp
from ncps.tf import LTCCell, CfCCell as WiredCfcCell

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if(filename.endswith('.csv')): print(filename)

data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv
data_out.csv
training_inputs_w_attitude.csv
data_in.csv
training_inputs.csv


In [5]:
# CSV_COLUMNS = ("vx", "vy", "vz", "omega_z")


# def load_image(img_path: str, img_shape: Tuple[int, int, int], reverse_channels: bool = True) -> Optional[ndarray]:
#     img = PIL.Image.open(img_path)
#     if img is not None:
#         # image shape is height, width, PIL takes width, height
#         resized = img.resize(img_shape[:2][::-1], PIL.Image.BILINEAR)
#         img_numpy = tf.keras.preprocessing.image.img_to_array(resized).astype(np.uint8)
#         if reverse_channels:
#             # reverse channels of image to match training
#             img_numpy = img_numpy[..., ::-1]

#         # add batch dim
#         img_numpy = np.expand_dims(img_numpy, axis=0)
#         return img_numpy
#     else:
#         return None


# def image_dir_generator(data_path: str, image_shape: Tuple[int, int, int], reverse_channels: bool = False):
#     """
#     Iterates through all of the pngs in the data folder, loads them as numpy array, and resizes them
#     to IMAGE_SHAPE. Yields the images using a generator for iteration
#     @param data_path: file path of dir with images
#     @param image_shape: height, width, channels tuple
#     @param reverse_channels: reverse image channels
#     """
#     contents = os.listdir(data_path)
#     contents = [os.path.join(data_path, c) for c in contents if 'png' in c]
#     contents.sort()
#     for path in contents:
#         img = load_image(img_path=path, img_shape=image_shape, reverse_channels=reverse_channels)
#         if img is not None:
#             yield img

In [6]:
# IMAGE_SHAPE = (144, 256, 3)
# labels = np.genfromtxt(os.path.join('/kaggle/input/data-1/1628106140.64', 'data_out.csv'), delimiter=',', skip_header=1)
# frames = list(image_dir_generator('/kaggle/input/data-1/1628106140.64', IMAGE_SHAPE))
# frame_stack_np = np.expand_dims(np.stack(frames, axis=0), axis=0)  # stack and add batch dim
# print(labels.shape)
# print(frame_stack_np.shape)

In [7]:
DROPOUT = 0.1

def generate_augmentation_layers(x, augmentation_params: Dict, single_step: bool):
    # translate -> rotate -> zoom -> noise
    trans = augmentation_params.get('translation', None)
    rot = augmentation_params.get('rotation', None)
    zoom = augmentation_params.get('zoom', None)
    noise = augmentation_params.get('noise', None)

    if trans is not None:
        x = wrap_time(keras.layers.experimental.preprocessing.RandomTranslation(
            height_factor=trans, width_factor=trans), single_step)(x)

    if rot is not None:
        x = wrap_time(keras.layers.experimental.preprocessing.RandomRotation(rot), single_step)(x)

    if zoom is not None:
        x = wrap_time(keras.layers.experimental.preprocessing.RandomZoom(
            height_factor=zoom, width_factor=zoom), single_step)(x)

    if noise:
        x = wrap_time(keras.layers.GaussianNoise(stddev=noise), single_step)(x)

    return x


def generate_normalization_layers(x, single_step: bool):
    rescaling_layer = keras.layers.experimental.preprocessing.Rescaling(1. / 255)

    normalization_layer = keras.layers.experimental.preprocessing.Normalization(
        mean=[0.41718618, 0.48529191, 0.38133072],
        variance=[.057, .05, .061])

    x = rescaling_layer(x)
    x = wrap_time(normalization_layer, single_step)(x)
    return x


def wrap_time(layer, single_step: bool):
    """
    Helper function that wraps layer in a timedistributed or not depending on the arguments of this function
    """
    if not single_step:
        return keras.layers.TimeDistributed(layer)
    else:
        return layer


def generate_network_trunk(seq_len,
                           image_shape,
                           augmentation_params: Dict = None,
                           batch_size=None,
                           single_step: bool = False,
                           no_norm_layer: bool = False, ):
    """
    Generates CNN image processing backbone used in all recurrent models. Uses Keras.Functional API

    returns input to be used in Keras.Model and x, a tensor that represents the output of the network that has shape
    (batch [None], seq_len, num_units) if single step is false and (batch [None], num_units) if single step is true.
    Input has shape (batch, h, w, c) if single step is True and (batch, seq, h, w, c) otherwise

    """

    if single_step:
        inputs_image = keras.Input(shape=image_shape, name="input_image")
    else:
        inputs_image = keras.Input(batch_input_shape=(batch_size, seq_len, *image_shape), name="input_image")

    xi = inputs_image

    if not no_norm_layer:
        xi = generate_normalization_layers(xi, single_step)

    if augmentation_params is not None:
        xi = generate_augmentation_layers(xi, augmentation_params, single_step)

    # Conv Layers
    xi = wrap_time(keras.layers.Conv2D(filters=24, kernel_size=(5, 5), strides=(2, 2), activation='relu'), single_step)(
        xi)
    xi = wrap_time(keras.layers.Conv2D(filters=36, kernel_size=(5, 5), strides=(2, 2), activation='relu'), single_step)(
        xi)
    xi = wrap_time(keras.layers.Conv2D(filters=48, kernel_size=(5, 5), strides=(2, 2), activation='relu'), single_step)(
        xi)
    xi = wrap_time(keras.layers.Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'), single_step)(
        xi)
    xi = wrap_time(keras.layers.Conv2D(filters=16, kernel_size=(3, 3), strides=(2, 2), activation='relu'), single_step)(
        xi)

    xi = wrap_time(keras.layers.Flatten(), single_step)(xi)
    xi = wrap_time(keras.layers.Dense(units=128, activation='linear'), single_step)(xi)
    xi = wrap_time(keras.layers.Dropout(rate=DROPOUT), single_step)(xi)

    # x = wrap_time(keras.layers.Concatenate(axis=-1), single_step)([xi, xp])
    # concatenate xi and xp using tf.concat along the last axis
    print(xi.shape, 'get_network_trunk')
    #x = wrap_time(keras.layers.Lambda(lambda y: tf.concat(y, axis=-1)), single_step)([xi, xp])
    print(xi.shape)

    return inputs_image, xi

In [8]:
class CTRNNCell(tf.keras.layers.Layer):
    def __init__(self, units, method, num_unfolds=None, tau=1, **kwargs):
        self.fixed_step_methods = {
            "euler": self.euler,
            "heun": self.heun,
            "rk4": self.rk4,
        }
        allowed_methods = ["euler", "heun", "rk4", "dopri5"]
        if not method in allowed_methods:
            raise ValueError(
                "Unknown ODE solver '{}', expected one of '{}'".format(
                    method, allowed_methods
                )
            )
        if method in self.fixed_step_methods.keys() and num_unfolds is None:
            raise ValueError(
                "Fixed-step ODE solver requires argument 'num_unfolds' to be specified!"
            )
        self.units = units
        self.state_size = units
        self.num_unfolds = num_unfolds
        self.method = method
        self.tau = tau
        super(CTRNNCell, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self.kernel = self.add_weight(
            shape=(input_dim, self.units), initializer="glorot_uniform", name="kernel"
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, self.units),
            initializer="orthogonal",
            name="recurrent_kernel",
        )
        self.bias = self.add_weight(
            shape=(self.units), initializer=tf.keras.initializers.Zeros(), name="bias"
        )
        self.scale = self.add_weight(
            shape=(self.units),
            initializer=tf.keras.initializers.Constant(1.0),
            name="scale",
        )
        if self.method == "dopri5":
            # Only load tfp packge if it is really needed
            import tensorflow_probability as tfp

            # We don't need the most precise solver to speed up training
            self.solver = tfp.math.ode.DormandPrince(
                rtol=0.01,
                atol=1e-04,
                first_step_size=0.01,
                safety_factor=0.8,
                min_step_size_factor=0.1,
                max_step_size_factor=10.0,
                max_num_steps=None,
                make_adjoint_solver_fn=None,
                validate_args=False,
                name="dormand_prince",
            )
        self.built = True

    def call(self, inputs, states):
        hidden_state = states[0]
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        if self.method == "dopri5":
            # Only load tfp packge if it is really needed
            import tensorflow_probability as tfp

            if not type(elapsed) == float:
                batch_dim = tf.shape(elapsed)[0]
                elapsed = tf.reshape(elapsed, [batch_dim])

                idx = tf.argsort(elapsed)
                solution_times = tf.gather(elapsed, idx)
            else:
                solution_times = tf.constant([elapsed])
            hidden_state = states[0]
            res = self.solver.solve(
                ode_fn=self.dfdt_wrapped,
                initial_time=0,
                initial_state=hidden_state,
                solution_times=solution_times,  # tfp.math.ode.ChosenBySolver(elapsed),
                constants={"input": inputs},
            )
            if not type(elapsed) == float:
                i2 = tf.stack([idx, tf.range(batch_dim)], axis=1)
                hidden_state = tf.gather_nd(res.states, i2)
            else:
                hidden_state = res.states[-1]
        else:
            delta_t = elapsed / self.num_unfolds
            method = self.fixed_step_methods[self.method]
            for i in range(self.num_unfolds):
                hidden_state = method(inputs, hidden_state, delta_t)
        return hidden_state, [hidden_state]

    def dfdt_wrapped(self, t, y, **constants):
        inputs = constants["input"]
        hidden_state = y
        return self.dfdt(inputs, hidden_state)

    def dfdt(self, inputs, hidden_state):
        h_in = tf.matmul(inputs, self.kernel)
        h_rec = tf.matmul(hidden_state, self.recurrent_kernel)
        dh_in = self.scale * tf.nn.tanh(h_in + h_rec + self.bias)
        if self.tau > 0:
            dh = dh_in - hidden_state * self.tau
        else:
            dh = dh_in
        return dh

    def euler(self, inputs, hidden_state, delta_t):
        dy = self.dfdt(inputs, hidden_state)
        return hidden_state + delta_t * dy

    def heun(self, inputs, hidden_state, delta_t):
        k1 = self.dfdt(inputs, hidden_state)
        k2 = self.dfdt(inputs, hidden_state + delta_t * k1)
        return hidden_state + delta_t * 0.5 * (k1 + k2)

    def rk4(self, inputs, hidden_state, delta_t):
        k1 = self.dfdt(inputs, hidden_state)
        k2 = self.dfdt(inputs, hidden_state + k1 * delta_t * 0.5)
        k3 = self.dfdt(inputs, hidden_state + k2 * delta_t * 0.5)
        k4 = self.dfdt(inputs, hidden_state + k3 * delta_t)

        return hidden_state + delta_t * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    
class LSTMCell(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = (units, units)
        self.initializer = "glorot_uniform"
        self.recurrent_initializer = "orthogonal"
        super(LSTMCell, self).__init__(**kwargs)

    def get_initial_state(self, inputs=None, batch_size=None, dtype=None):
        return (
            tf.zeros([batch_size, self.units], dtype=tf.float32),
            tf.zeros([batch_size, self.units], dtype=tf.float32),
        )

    def build(self, input_shape):
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_shape = (input_shape[0][-1] + input_shape[1][-1],)

        # name weights with _lstm suffix so parents with this and other rnns can save weights and not have name collide
        self.input_kernel = self.add_weight(
            shape=(input_shape[-1], 4 * self.units),
            initializer=self.initializer,
            name="input_kernel_lstm",
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, 4 * self.units),
            initializer=self.recurrent_initializer,
            name="recurrent_kernel_lstm",
        )
        self.bias = self.add_weight(
            shape=(4 * self.units),
            initializer=tf.keras.initializers.Zeros(),
            name="bias_lstm",
        )

        self.built = True

    def call(self, inputs, states):
        cell_state, output_state = states
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            if isinstance(elapsed, float):
                # tensors should be same shape to concat
                elapsed = tf.constant(elapsed)
                batch_dim = tf.shape(inputs[0])[0]  # can't use .shape, need to use tf.shape()
                elapsed = tf.reshape(elapsed, (1, 1))
                elapsed = tf.repeat(elapsed, repeats=batch_dim, axis=0)
            inputs = tf.concat([inputs[0], elapsed], axis=-1)

        z = (
                tf.matmul(inputs, self.input_kernel)
                + tf.matmul(output_state, self.recurrent_kernel)
                + self.bias
        )
        i, ig, fg, og = tf.split(z, 4, axis=-1)

        input_activation = tf.nn.tanh(i)
        input_gate = tf.nn.sigmoid(ig)
        forget_gate = tf.nn.sigmoid(fg + 1.0)
        output_gate = tf.nn.sigmoid(og)

        new_cell = cell_state * forget_gate + input_activation * input_gate
        output_state = tf.nn.tanh(new_cell) * output_gate

        return output_state, [new_cell, output_state]


# mmRNN uses a LSTM as memory cell and CT-RNN (= neural ODE) for the time-continuous pathway
class mmRNN(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = (units, units)
        self.initializer = "glorot_uniform"
        self.recurrent_initializer = "orthogonal"
        self.ctrnn = CTRNNCell(self.units, num_unfolds=4, method="euler")
        super(mmRNN, self).__init__(**kwargs)

    def get_initial_state(self, inputs=None, batch_size=None, dtype=None):
        return (
            tf.zeros([batch_size, self.units], dtype=tf.float32),
            tf.zeros([batch_size, self.units], dtype=tf.float32),
        )

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self.ctrnn.build([self.units])
        # name weights with _mmrnn suffix so don't have same weight names as child CTRNNCell which also has same weights
        self.input_kernel = self.add_weight(
            shape=(input_dim, 4 * self.units),
            initializer=self.initializer,
            name="input_kernel_mmrnn",
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, 4 * self.units),
            initializer=self.recurrent_initializer,
            name="recurrent_kernel_mmrnn",
        )
        self.bias = self.add_weight(
            shape=(4 * self.units),
            initializer=tf.keras.initializers.Zeros(),
            name="bias_mmrn",
        )

        self.built = True

    def call(self, inputs, states):
        cell_state, ode_state = states
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        z = (
                tf.matmul(inputs, self.input_kernel)
                + tf.matmul(ode_state, self.recurrent_kernel)
                + self.bias
        )
        i, ig, fg, og = tf.split(z, 4, axis=-1)

        input_activation = tf.nn.tanh(i)
        input_gate = tf.nn.sigmoid(ig)
        forget_gate = tf.nn.sigmoid(fg + 3.0)
        output_gate = tf.nn.sigmoid(og)

        new_cell = cell_state * forget_gate + input_activation * input_gate
        ode_input = tf.nn.tanh(new_cell) * output_gate

        # Implementation choice on how to parametrize ODE component
        ode_output, new_ode_state = self.ctrnn.call([ode_input, elapsed], [ode_state])
        # ode_output, new_ode_state = self.ctrnn.call([ode_input, elapsed], [ode_input])

        return ode_output, [new_cell, new_ode_state[0]]


class CTGRU(tf.keras.layers.Layer):
    # https://arxiv.org/abs/1710.04110
    def __init__(self, units, M=8, **kwargs):
        self.units = units
        self.M = M
        self.state_size = units * self.M

        # Pre-computed tau table (as recommended in paper)
        self.ln_tau_table = np.empty(self.M)
        self.tau_table = np.empty(self.M)
        tau = 1.0
        for i in range(self.M):
            self.ln_tau_table[i] = np.log(tau)
            self.tau_table[i] = tau
            tau = tau * (10.0 ** 0.5)

        super(CTGRU, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self.retrieval_layer = tf.keras.layers.Dense(
            self.units * self.M, activation=None
        )
        self.detect_layer = tf.keras.layers.Dense(self.units, activation="tanh")
        self.update_layer = tf.keras.layers.Dense(self.units * self.M, activation=None)
        self.built = True

    def call(self, inputs, states):
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1].astype(np.float32)
            inputs = inputs[0].astype(np.float32)

        batch_dim = tf.shape(inputs)[0]

        # States is actually 2D
        h_hat = tf.reshape(states[0], [batch_dim, self.units, self.M])
        h = tf.reduce_sum(h_hat, axis=2)
        states = None  # Set state to None, to avoid misuses (bugs) in the code below

        # Retrieval
        fused_input = tf.concat([inputs, h], axis=-1)
        ln_tau_r = self.retrieval_layer(fused_input)
        ln_tau_r = tf.reshape(ln_tau_r, shape=[batch_dim, self.units, self.M])
        sf_input_r = -tf.square(ln_tau_r - self.ln_tau_table)
        rki = tf.nn.softmax(logits=sf_input_r, axis=2)

        q_input = tf.reduce_sum(rki * h_hat, axis=2)
        reset_value = tf.concat([inputs, q_input], axis=1)
        qk = self.detect_layer(reset_value)
        qk = tf.reshape(qk, [batch_dim, self.units, 1])  # in order to broadcast

        ln_tau_s = self.update_layer(fused_input)
        ln_tau_s = tf.reshape(ln_tau_s, shape=[batch_dim, self.units, self.M])
        sf_input_s = -tf.square(ln_tau_s - self.ln_tau_table)
        ski = tf.nn.softmax(logits=sf_input_s, axis=2)

        # Now the elapsed time enters the state update
        base_term = (1 - ski) * h_hat + ski * qk
        exp_term = tf.exp(-elapsed / self.tau_table)
        exp_term = tf.reshape(exp_term, [1, 1, self.M])  # reshape to add batch dim
        exp_term = tf.repeat(exp_term, repeats=[batch_dim], axis=0)  # repeat for each element of batch
        h_hat_next = base_term * tf.cast(exp_term, dtype=tf.float32)

        # Compute new state
        h_next = tf.reduce_sum(h_hat_next, axis=2)
        h_hat_next_flat = tf.reshape(h_hat_next, shape=[batch_dim, self.units * self.M])
        return h_next, [h_hat_next_flat]


class VanillaRNN(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = units

        super(VanillaRNN, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self._layer = tf.keras.layers.Dense(self.units, activation="tanh")
        self._out_layer = tf.keras.layers.Dense(self.units, activation=None)
        self._tau = self.add_weight(
            "tau",
            shape=(self.units),
            dtype=tf.float32,
            initializer=tf.keras.initializers.Constant(0.1),
        )
        self.built = True

    def call(self, inputs, states):
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        fused_input = tf.concat([inputs, states[0]], axis=-1)
        new_states = self._out_layer(self._layer(fused_input)) - elapsed * self._tau

        return new_states, [new_states]


class BidirectionalRNN(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = (units, units, units)

        self.ctrnn = CTRNNCell(self.units, num_unfolds=4, method="euler")
        self.lstm = LSTMCell(units=self.units)

        super(BidirectionalRNN, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]
        self._out_layer = tf.keras.layers.Dense(self.units, activation=None)
        fused_dim = ((input_dim + self.units,), (1,))
        self.lstm.build(fused_dim)
        self.ctrnn.build(fused_dim)
        self.built = True

    def call(self, inputs, states):
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        lstm_state = [states[0], states[1]]
        lstm_input = [tf.concat([inputs, states[2]], axis=-1), elapsed]
        ctrnn_state = [states[2]]
        ctrnn_input = [tf.concat([inputs, states[1]], axis=-1), elapsed]

        lstm_out, new_lstm_states = self.lstm.call(lstm_input, lstm_state)
        ctrnn_out, new_ctrnn_state = self.ctrnn.call(ctrnn_input, ctrnn_state)

        fused_output = lstm_out + ctrnn_out
        return (
            fused_output,
            [new_lstm_states[0], new_lstm_states[1], new_ctrnn_state[0]],
        )


class GRUD(tf.keras.layers.Layer):
    # Implemented according to
    # https://www.nature.com/articles/s41598-018-24271-9.pdf
    # without the masking

    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = units
        super(GRUD, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self._reset_gate = tf.keras.layers.Dense(
            self.units, activation="sigmoid", kernel_initializer="glorot_uniform"
        )
        self._detect_signal = tf.keras.layers.Dense(
            self.units, activation="tanh", kernel_initializer="glorot_uniform"
        )
        self._update_gate = tf.keras.layers.Dense(
            self.units, activation="sigmoid", kernel_initializer="glorot_uniform"
        )
        self._d_gate = tf.keras.layers.Dense(
            self.units, activation="relu", kernel_initializer="glorot_uniform"
        )

        self.built = True

    def call(self, inputs, states):
        # d_gate needs elapsed to have 2 dims
        batch_dim = tf.shape(inputs)[0]
        elapsed = tf.ones((batch_dim, 1))
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        dt = self._d_gate(elapsed)
        gamma = tf.exp(-dt)
        h_hat = states[0] * gamma

        fused_input = tf.concat([inputs, h_hat], axis=-1)
        rt = self._reset_gate(fused_input)
        zt = self._update_gate(fused_input)

        reset_value = tf.concat([inputs, rt * h_hat], axis=-1)
        h_tilde = self._detect_signal(reset_value)

        # Compute new state
        ht = zt * h_hat + (1.0 - zt) * h_tilde

        return ht, [ht]


class PhasedLSTM(tf.keras.layers.Layer):
    # Implemented according to
    # https://papers.nips.cc/paper/6310-phased-lstm-accelerating-recurrent-network-training-for-long-or-event-based-sequences.pdf

    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = (units, units)
        self.initializer = "glorot_uniform"
        self.recurrent_initializer = "orthogonal"
        super(PhasedLSTM, self).__init__(**kwargs)

    def get_initial_state(self, inputs=None, batch_size=None, dtype=None):
        return (
            tf.zeros([batch_size, self.units], dtype=tf.float32),
            tf.zeros([batch_size, self.units], dtype=tf.float32),
        )

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self.input_kernel = self.add_weight(
            shape=(input_dim, 4 * self.units),
            initializer=self.initializer,
            name="input_kernel",
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, 4 * self.units),
            initializer=self.recurrent_initializer,
            name="recurrent_kernel",
        )
        self.bias = self.add_weight(
            shape=(4 * self.units),
            initializer=tf.keras.initializers.Zeros(),
            name="bias",
        )
        self.tau = self.add_weight(
            shape=(1,), initializer=tf.keras.initializers.Zeros(), name="tau"
        )
        self.ron = self.add_weight(
            shape=(1,), initializer=tf.keras.initializers.Zeros(), name="ron"
        )
        self.s = self.add_weight(
            shape=(1,), initializer=tf.keras.initializers.Zeros(), name="s"
        )

        self.built = True

    def call(self, inputs, states):
        cell_state, hidden_state = states
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        # Leaky constant taken fromt he paper
        alpha = 0.001
        # Make sure these values are positive
        tau = tf.nn.softplus(self.tau)
        s = tf.nn.softplus(self.s)
        ron = tf.nn.softplus(self.ron)

        phit = tf.math.mod(elapsed - s, tau) / tau
        kt = tf.where(
            tf.less(phit, 0.5 * ron),
            2 * phit * ron,
            tf.where(tf.less(phit, ron), 2.0 - 2 * phit / ron, alpha * phit),
        )

        z = (
                tf.matmul(inputs, self.input_kernel)
                + tf.matmul(hidden_state, self.recurrent_kernel)
                + self.bias
        )
        i, ig, fg, og = tf.split(z, 4, axis=-1)

        input_activation = tf.nn.tanh(i)
        input_gate = tf.nn.sigmoid(ig)
        forget_gate = tf.nn.sigmoid(fg + 1.0)
        output_gate = tf.nn.sigmoid(og)

        c_tilde = cell_state * forget_gate + input_activation * input_gate
        c = kt * c_tilde + (1.0 - kt) * cell_state

        h_tilde = tf.nn.tanh(c_tilde) * output_gate
        h = kt * h_tilde + (1.0 - kt) * hidden_state

        return h, [c, h]


class GRUODE(tf.keras.layers.Layer):
    # Implemented according to
    # https://arxiv.org/pdf/1905.12374.pdf
    # without the Bayesian stuff

    def __init__(self, units, num_unfolds=4, **kwargs):
        self.units = units
        self.num_unfolds = num_unfolds
        self.state_size = units
        super(GRUODE, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]
        self._reset_gate = tf.keras.layers.Dense(
            self.units,
            activation="sigmoid",
            bias_initializer=tf.constant_initializer(1),
        )
        self._detect_signal = tf.keras.layers.Dense(self.units, activation="tanh")
        self._update_gate = tf.keras.layers.Dense(self.units, activation="sigmoid")

        self.built = True

    def _dh_dt(self, inputs, states):
        fused_input = tf.concat([inputs, states], axis=-1)
        rt = self._reset_gate(fused_input)
        zt = self._update_gate(fused_input)

        reset_value = tf.concat([inputs, rt * states], axis=-1)
        gt = self._detect_signal(reset_value)

        # Compute new state
        dhdt = (1.0 - zt) * (gt - states)
        return dhdt

    def euler(self, inputs, hidden_state, delta_t):
        dy = self._dh_dt(inputs, hidden_state)
        return hidden_state + delta_t * dy

    def call(self, inputs, states):
        elapsed = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        delta_t = elapsed / self.num_unfolds
        hidden_state = states[0]
        for i in range(self.num_unfolds):
            hidden_state = self.euler(inputs, hidden_state, delta_t)
        return hidden_state, [hidden_state]

        return ht, [ht]


class HawkLSTMCell(tf.keras.layers.Layer):
    # https://papers.nips.cc/paper/7252-the-neural-hawkes-process-a-neurally-self-modulating-multivariate-point-process.pdf
    def __init__(self, units, **kwargs):
        self.units = units
        self.state_size = (units, units, units)  # state is a tripple
        self.initializer = "glorot_uniform"
        self.recurrent_initializer = "orthogonal"
        super(HawkLSTMCell, self).__init__(**kwargs)

    def get_initial_state(self, inputs=None, batch_size=None, dtype=None):
        return (
            tf.zeros([batch_size, self.units], dtype=tf.float32),
            tf.zeros([batch_size, self.units], dtype=tf.float32),
            tf.zeros([batch_size, self.units], dtype=tf.float32),
        )

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]
        self.input_kernel = self.add_weight(
            shape=(input_dim, 7 * self.units),
            initializer=self.initializer,
            name="input_kernel",
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, 7 * self.units),
            initializer=self.recurrent_initializer,
            name="recurrent_kernel",
        )
        self.bias = self.add_weight(
            shape=(7 * self.units),
            initializer=tf.keras.initializers.Zeros(),
            name="bias",
        )

        self.built = True

    def call(self, inputs, states):
        c, c_bar, h = states
        # assume that input is k and that elapsed is always 1
        # k = inputs[0]  # Is the input
        # delta_t = inputs[1]  # is the elapsed time
        k = inputs
        delta_t = 1.0
        z = (
                tf.matmul(k, self.input_kernel)
                + tf.matmul(h, self.recurrent_kernel)
                + self.bias
        )
        i, ig, fg, og, ig_bar, fg_bar, d = tf.split(z, 7, axis=-1)

        input_activation = tf.nn.tanh(i)
        input_gate = tf.nn.sigmoid(ig)
        input_gate_bar = tf.nn.sigmoid(ig_bar)
        forget_gate = tf.nn.sigmoid(fg)
        forget_gate_bar = tf.nn.sigmoid(fg_bar)
        output_gate = tf.nn.sigmoid(og)
        delta_gate = tf.nn.softplus(d)

        new_c = c * forget_gate + input_activation * input_gate
        new_c_bar = c_bar * forget_gate_bar + input_activation * input_gate_bar

        c_t = new_c_bar + (new_c - new_c_bar) * tf.exp(-delta_gate * delta_t)
        output_state = tf.nn.tanh(c_t) * output_gate

        return output_state, [new_c, new_c_bar, output_state]

In [9]:
def lecun_tanh(x):
    return 1.7159 * tf.nn.tanh(0.666 * x)


class CfcCell(tf.keras.layers.Layer):
    def __init__(self, units, hparams, **kwargs):
        super(CfcCell, self).__init__(**kwargs)
        self.units = units
        self.state_size = units
        self.hparams = hparams
        self._no_gate = False

    def build(self, input_shape):
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]
        else:
            input_dim = input_shape[-1]

        if self.hparams["backbone_activation"] == "silu":
            backbone_activation = tf.nn.silu
        elif self.hparams["backbone_activation"] == "relu":
            backbone_activation = tf.nn.relu
        elif self.hparams["backbone_activation"] == "tanh":
            backbone_activation = tf.nn.tanh
        elif self.hparams["backbone_activation"] == "lecun":
            backbone_activation = lecun_tanh
        elif self.hparams["backbone_activation"] == "softplus":
            backbone_activation = tf.nn.softplus
        else:
            raise ValueError("Unknown backbone activation")

        self._no_gate = False
        if "no_gate" in self.hparams:
            self._no_gate = self.hparams["no_gate"]
        self._minimal = False
        if "minimal" in self.hparams:
            self._minimal = self.hparams["minimal"]

        self.backbone = []
        for i in range(self.hparams["backbone_layers"]):

            self.backbone.append(
                tf.keras.layers.Dense(
                    self.hparams["backbone_units"],
                    backbone_activation,
                    kernel_regularizer=tf.keras.regularizers.L2(
                        self.hparams["weight_decay"]
                    ),
                )
            )
            self.backbone.append(tf.keras.layers.Dropout(self.hparams["backbone_dr"]))

        self.backbone = tf.keras.models.Sequential(self.backbone)

        if self._minimal:
            self.ff1 = tf.keras.layers.Dense(
                self.units,
                kernel_regularizer=tf.keras.regularizers.L2(
                    self.hparams["weight_decay"]
                ),
            )
            self.w_tau = self.add_weight(
                shape=(1, self.units), initializer=tf.keras.initializers.Zeros()
            )
            self.A = self.add_weight(
                shape=(1, self.units), initializer=tf.keras.initializers.Ones()
            )
        else:
            self.ff1 = tf.keras.layers.Dense(
                self.units,
                lecun_tanh,
                kernel_regularizer=tf.keras.regularizers.L2(
                    self.hparams["weight_decay"]
                ),
            )
            self.ff2 = tf.keras.layers.Dense(
                self.units,
                lecun_tanh,
                kernel_regularizer=tf.keras.regularizers.L2(
                    self.hparams["weight_decay"]
                ),
            )
            self.time_a = tf.keras.layers.Dense(
                self.units,
                kernel_regularizer=tf.keras.regularizers.L2(
                    self.hparams["weight_decay"]
                ),
            )
            self.time_b = tf.keras.layers.Dense(
                self.units,
                kernel_regularizer=tf.keras.regularizers.L2(
                    self.hparams["weight_decay"]
                ),
            )
        self.built = True

    def call(self, inputs, states, **kwargs):
        hidden_state = states[0]
        t = 1.0
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            t = tf.reshape(elapsed, [-1, 1])
            inputs = inputs[0]

        x = tf.keras.layers.Concatenate()([inputs, hidden_state])
        x = self.backbone(x)
        ff1 = self.ff1(x)
        if self._minimal:
            # Solution
            new_hidden = (
                -self.A
                * tf.math.exp(-t * (tf.math.abs(self.w_tau) + tf.math.abs(ff1)))
                * ff1
                + self.A
            )
        else:
            # Cfc
            ff2 = self.ff2(x)
            t_a = self.time_a(x)
            t_b = self.time_b(x)
            t_interp = tf.nn.sigmoid(-t_a * t + t_b)
            if self._no_gate:
                new_hidden = ff1 + t_interp * ff2
            else:
                new_hidden = ff1 * (1.0 - t_interp) + t_interp * ff2

        return new_hidden, [new_hidden]


class MixedCfcCell(tf.keras.layers.Layer):
    def __init__(self, units, hparams, **kwargs):
        self.units = units
        self.state_size = (units, units)
        self.initializer = "glorot_uniform"
        self.recurrent_initializer = "orthogonal"
        self.forget_gate_bias = 1
        if "forget_bias" in hparams.keys():
            self.forget_gate_bias = hparams["forget_bias"]
        self.cfc = CfcCell(self.units, hparams)
        super(MixedCfcCell, self).__init__(**kwargs)

    def get_initial_state(self, inputs=None, batch_size=None, dtype=None):
        return (
            tf.zeros([batch_size, self.units], dtype=tf.float32),
            tf.zeros([batch_size, self.units], dtype=tf.float32),
        )

    def build(self, input_shape):
        input_dim = input_shape[-1]
        if isinstance(input_shape[0], tuple):
            # Nested tuple
            input_dim = input_shape[0][-1]

        self.cfc.build(input_shape)
        self.input_kernel = self.add_weight(
            shape=(input_dim, 4 * self.units),
            initializer=self.initializer,
            name="input_kernel",
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, 4 * self.units),
            initializer=self.recurrent_initializer,
            name="recurrent_kernel",
        )
        self.bias = self.add_weight(
            shape=(4 * self.units),
            initializer=tf.keras.initializers.Zeros(),
            name="bias",
        )

        self.built = True

    def call(self, inputs, states, **kwargs):
        cell_state, ode_state = states
        elapsed = tf.zeros((1,), dtype=tf.float32)
        if (isinstance(inputs, tuple) or isinstance(inputs, list)) and len(inputs) > 1:
            elapsed = inputs[1]
            inputs = inputs[0]

        z = (
            tf.matmul(inputs, self.input_kernel)
            + tf.matmul(ode_state, self.recurrent_kernel)
            + self.bias
        )
        i, ig, fg, og = tf.split(z, 4, axis=-1)

        input_activation = tf.nn.tanh(i)
        input_gate = tf.nn.sigmoid(ig)
        forget_gate = tf.nn.sigmoid(fg + self.forget_gate_bias)
        output_gate = tf.nn.sigmoid(og)

        new_cell = cell_state * forget_gate + input_activation * input_gate
        ode_input = tf.nn.tanh(new_cell) * output_gate  # LSTM output = ODE input

        # Implementation choice on how to parametrize ODE component
        ode_output, new_ode_state = self.cfc([ode_input, elapsed], [ode_state])
        # ode_output, new_ode_state = self.ctrnn.call([ode_input, elapsed], [ode_input])

        return ode_output, [new_cell, new_ode_state[0]]


class CFCLTCCell(tf.keras.layers.AbstractRNNCell):
    def __init__(self, units, ode_unfolds=3, epsilon=1e-8, **kwargs):

        self.units = units
        self._init_ranges = {
            "gleak": (0.001, 1.0),
            "vleak": (-0.2, 0.2),
            "cm": (0.4, 0.6),
            "w": (0.001, 1.0),
            "sigma": (3, 8),
            "mu": (0.3, 0.8),
            "sensory_w": (0.001, 1.0),
            "sensory_sigma": (3, 8),
            "sensory_mu": (0.3, 0.8),
        }

        self._ode_unfolds = ode_unfolds
        self._epsilon = epsilon
        super(LTCCell, self).__init__(name="ltc_cell")

    @property
    def state_size(self):
        return self.units

    @property
    def sensory_size(self):
        return self.input_dim

    def _get_initializer(self, param_name):
        minval, maxval = self._init_ranges[param_name]
        if minval == maxval:
            return tf.keras.initializers.Constant(minval)
        else:
            return tf.keras.initializers.RandomUniform(minval, maxval)

    def _erev_initializer(self, shape=None, dtype=None):
        return np.random.default_rng().choice([-1, 1], size=shape)

    def build(self, input_shape):

        # Check if input_shape is nested tuple/list
        if isinstance(input_shape[0], (tuple, list)):
            input_shape = input_shape[0]

        self.input_dim = input_shape[-1]

        self._params = {}
        self._params["gleak"] = self.add_weight(
            name="gleak",
            shape=(self.state_size,),
            dtype=tf.float32,
            constraint=tf.keras.constraints.NonNeg(),
            initializer=self._get_initializer("gleak"),
        )
        self._params["vleak"] = self.add_weight(
            name="vleak",
            shape=(self.state_size,),
            dtype=tf.float32,
            initializer=self._get_initializer("vleak"),
        )
        self._params["cm"] = self.add_weight(
            name="cm",
            shape=(self.state_size,),
            dtype=tf.float32,
            constraint=tf.keras.constraints.NonNeg(),
            initializer=self._get_initializer("cm"),
        )
        self._params["sigma"] = self.add_weight(
            name="sigma",
            shape=(self.state_size, self.state_size),
            dtype=tf.float32,
            initializer=self._get_initializer("sigma"),
        )
        self._params["mu"] = self.add_weight(
            name="mu",
            shape=(self.state_size, self.state_size),
            dtype=tf.float32,
            initializer=self._get_initializer("mu"),
        )
        self._params["w"] = self.add_weight(
            name="w",
            shape=(self.state_size, self.state_size),
            dtype=tf.float32,
            constraint=tf.keras.constraints.NonNeg(),
            initializer=self._get_initializer("w"),
        )
        self._params["erev"] = self.add_weight(
            name="erev",
            shape=(self.state_size, self.state_size),
            dtype=tf.float32,
            initializer=self._erev_initializer,
        )

        self._params["sensory_sigma"] = self.add_weight(
            name="sensory_sigma",
            shape=(self.sensory_size, self.state_size),
            dtype=tf.float32,
            initializer=self._get_initializer("sensory_sigma"),
        )
        self._params["sensory_mu"] = self.add_weight(
            name="sensory_mu",
            shape=(self.sensory_size, self.state_size),
            dtype=tf.float32,
            initializer=self._get_initializer("sensory_mu"),
        )
        self._params["sensory_w"] = self.add_weight(
            name="sensory_w",
            shape=(self.sensory_size, self.state_size),
            dtype=tf.float32,
            constraint=tf.keras.constraints.NonNeg(),
            initializer=self._get_initializer("sensory_w"),
        )
        self._params["sensory_erev"] = self.add_weight(
            name="sensory_erev",
            shape=(self.sensory_size, self.state_size),
            dtype=tf.float32,
            initializer=self._erev_initializer,
        )

        self._params["input_w"] = self.add_weight(
            name="input_w",
            shape=(self.sensory_size,),
            dtype=tf.float32,
            initializer=tf.keras.initializers.Constant(1),
        )
        self._params["input_b"] = self.add_weight(
            name="input_b",
            shape=(self.sensory_size,),
            dtype=tf.float32,
            initializer=tf.keras.initializers.Constant(0),
        )

        self._params["output_w"] = self.add_weight(
            name="output_w",
            shape=(self.state_size,),
            dtype=tf.float32,
            initializer=tf.keras.initializers.Constant(1),
        )
        self._params["output_b"] = self.add_weight(
            name="output_b",
            shape=(self.state_size,),
            dtype=tf.float32,
            initializer=tf.keras.initializers.Constant(0),
        )
        self.built = True

    def _sigmoid(self, v_pre, mu, sigma):
        v_pre = tf.expand_dims(v_pre, axis=-1)  # For broadcasting
        mues = v_pre - mu
        x = sigma * mues
        return tf.nn.sigmoid(x)

    def _ode_solver(self, inputs, state, elapsed_time):
        v_pre = state

        # We can pre-compute the effects of the sensory neurons here
        sensory_w_activation = self._params["sensory_w"] * self._sigmoid(
            inputs, self._params["sensory_mu"], self._params["sensory_sigma"]
        )

        sensory_rev_activation = sensory_w_activation * self._params["sensory_erev"]

        # Reduce over dimension 1 (=source sensory neurons)
        w_numerator_sensory = tf.reduce_sum(sensory_rev_activation, axis=1)
        w_denominator_sensory = tf.reduce_sum(sensory_w_activation, axis=1)

        # cm/t is loop invariant
        cm_t = self._params["cm"] / tf.cast(
            (elapsed_time + 1e-3) / self._ode_unfolds, dtype=tf.float32
        )

        # Unfold the multiply ODE multiple times into one RNN step
        for t in range(self._ode_unfolds):
            w_activation = self._params["w"] * self._sigmoid(
                v_pre, self._params["mu"], self._params["sigma"]
            )

            rev_activation = w_activation * self._params["erev"]

            # Reduce over dimension 1 (=source neurons)
            w_numerator = tf.reduce_sum(rev_activation, axis=1) + w_numerator_sensory
            w_denominator = tf.reduce_sum(w_activation, axis=1) + w_denominator_sensory

            numerator = (
                cm_t * v_pre
                + self._params["gleak"] * self._params["vleak"]
                + w_numerator
            )
            denominator = cm_t + self._params["gleak"] + w_denominator

            # Avoid dividing by 0
            v_pre = numerator / (denominator + self._epsilon)

        return v_pre

    def _map_inputs(self, inputs):
        inputs = inputs * self._params["input_w"]
        inputs = inputs + self._params["input_b"]
        return inputs

    def _map_outputs(self, state):
        output = state
        output = output * self._params["output_w"]
        output = output + self._params["output_b"]
        return output

    def call(self, inputs, states):
        if isinstance(inputs, (tuple, list)):
            # Irregularly sampled mode
            inputs, elapsed_time = inputs
        else:
            # Regularly sampled mode (elapsed time = 1 second)
            elapsed_time = 1.0
        inputs = self._map_inputs(inputs)

        next_state = self._ode_solver(inputs, states[0], elapsed_time)

        outputs = self._map_outputs(next_state)

        return outputs, [next_state]

In [10]:
DEFAULT_NCP_SEED = 22222
DEFAULT_CFC_CONFIG = {
    "clipnorm": 1,
    "backbone_activation": "silu",
    "backbone_dr": 0.1,
    "forget_bias": 1.6,
    "backbone_units": 128,
    "backbone_layers": 1,
    "weight_decay": 1e-06
}

def generate_ncp_model(seq_len,
                       image_shape,
                       augmentation_params=None,
                       batch_size=None,
                       seed=DEFAULT_NCP_SEED,
                       single_step: bool = False,
                       no_norm_layer: bool = False,
                       ):
    inputs_image, x = generate_network_trunk(
        seq_len,
        image_shape,
        augmentation_params=augmentation_params,
        batch_size=batch_size,
        single_step=single_step,
        no_norm_layer=no_norm_layer,
    )

    # Setup the network
    wiring = kncp.wirings.NCP(
        inter_neurons=18,  # Number of inter neurons
        command_neurons=12,  # Number of command neurons
        motor_neurons=4,  # Number of motor neurons
        sensory_fanout=6,  # How many outgoing synapses has each sensory neuron
        inter_fanout=4,  # How many outgoing synapses has each inter neuron
        recurrent_command_synapses=4,  # Now many recurrent synapses are in the
        # command neuron layer
        motor_fanin=6,  # How many incoming syanpses has each motor neuron,
        seed=seed,  # random seed to generate connections between nodes
    )

    rnn_cell = LTCCell(wiring)

    if single_step:
        inputs_state = tf.keras.Input(shape=(rnn_cell.state_size,))
        # wrap output states in list since want output to just be ndarray, not list of 1 el ndarray
        motor_out, [output_states] = rnn_cell(x, inputs_state)
        ncp_model = keras.Model([inputs_image, inputs_state], [motor_out, output_states])
    else:
        x = keras.layers.RNN(rnn_cell,
                             batch_input_shape=(batch_size,
                                                seq_len,
                                                x.shape[-1]),
                             return_sequences=True)(x)

        ncp_model = keras.Model([inputs_image], [x])

    return ncp_model

def generate_lstm_model(
        rnn_sizes,
        seq_len,
        image_shape,
        dropout=0.1,
        recurrent_dropout=0.1,
        rnn_stateful=False,
        batch_size=None,
        augmentation_params=None,
        single_step: bool = False,
        no_norm_layer: bool = False,
):
    inputs_image, x = generate_network_trunk(
        seq_len,
        image_shape,
        augmentation_params=augmentation_params,
        batch_size=batch_size,
        single_step=single_step,
        no_norm_layer=no_norm_layer,
    )

    # vars for single step model
    c_inputs = []
    h_inputs = []
    c_outputs = []
    h_outputs = []
    for (ix, s) in enumerate(rnn_sizes):
        if single_step:
            rnn_cell = tf.keras.layers.LSTMCell(s)
            # keep track of input for each layer of rnn
            c_input = tf.keras.Input(shape=(rnn_cell.state_size[0]))
            h_input = tf.keras.Input(shape=(rnn_cell.state_size[1]))

            x, [c_state, h_state] = rnn_cell(x, [c_input, h_input])
            c_inputs.append(c_input)
            h_inputs.append(h_input)
            c_outputs.append(c_state)
            h_outputs.append(h_state)
        else:
            x = keras.layers.LSTM(
                s,
                batch_input_shape=(
                    batch_size,
                    seq_len,
                    x.shape[-1]
                ),
                return_sequences=True,
                stateful=rnn_stateful,
                dropout=dropout,
                recurrent_dropout=recurrent_dropout
            )(x)

    x = keras.layers.Dense(units=4, activation='linear')(x)
    if single_step:
        lstm_model = keras.Model([inputs_image, *c_inputs, *h_inputs], [x, *c_outputs, *h_outputs])
    else:
        lstm_model = keras.Model([inputs_image], [x])

    return lstm_model

def generate_ctrnn_model(rnn_sizes,
                         seq_len,
                         image_shape,
                         augmentation_params=None,
                         rnn_stateful=False,
                         batch_size=None,
                         ct_network_type='ctrnn',
                         config=DEFAULT_CFC_CONFIG,
                         single_step: bool = False,
                         no_norm_layer: bool = False,
                         **kwargs,
                         ):
    inputs_image, x = generate_network_trunk(
        seq_len,
        image_shape,
        augmentation_params=augmentation_params,
        batch_size=batch_size,
        single_step=single_step,
        no_norm_layer=no_norm_layer,
    )

    # vars for single step model
    all_hidden_inputs = []  # shape: num layers x num hidden x hidden size
    all_hidden_outputs = []
    for (ix, s) in enumerate(rnn_sizes):
        if ct_network_type == 'ctrnn':
            rnn_cell = CTRNNCell(units=s, method='dopri5')
        elif ct_network_type == "node":
            rnn_cell = CTRNNCell(units=s, method="dopri5", tau=0)
        elif ct_network_type == "mmrnn":
            rnn_cell = mmRNN(units=s)
        elif ct_network_type == "ctgru":
            rnn_cell = CTGRU(units=s)
        elif ct_network_type == "vanilla":
            rnn_cell = VanillaRNN(units=s)
        elif ct_network_type == "bidirect":
            rnn_cell = BidirectionalRNN(units=s)
        elif ct_network_type == "grud":
            rnn_cell = GRUD(units=s)
        elif ct_network_type == "phased":
            rnn_cell = PhasedLSTM(units=s)
        elif ct_network_type == "gruode":
            rnn_cell = GRUODE(units=s)
        elif ct_network_type == "hawk":
            rnn_cell = HawkLSTMCell(units=s)
        elif ct_network_type == "ltc":
            rnn_cell = CFCLTCCell(units=s)
        elif ct_network_type == "cfc":
            rnn_cell = CfcCell(units=s, hparams=config)
        elif ct_network_type == "mixedcfc":
            rnn_cell = MixedCfcCell(units=s, hparams=config)
        elif ct_network_type == "wiredcfccell":
            wiring = kncp.wirings.NCP(
                inter_neurons=18,  # Number of inter neurons
                command_neurons=12,  # Number of command neurons
                motor_neurons=4,  # Number of motor neurons
                sensory_fanout=6,  # How many outgoing synapses has each sensory neuron
                inter_fanout=4,  # How many outgoing synapses has each inter neuron
                recurrent_command_synapses=4,  # Now many recurrent synapses are in the
                # command neuron
                # layer
                motor_fanin=6,  # How many incoming syanpses has each motor neuron,
                seed=kwargs.get("wiredcfc_seed", DEFAULT_NCP_SEED),  # random seed to generate connections between nodes
            )
            rnn_cell = WiredCfcCell(wiring=wiring, mode="default")
        else:
            raise ValueError("Unknown model type '{}'".format(ct_network_type))

        if single_step:
            # keep track of input for each layer of rnn
            if isinstance(rnn_cell.state_size, int):
                # only 1 hidden state
                hidden_inputs = [tf.keras.Input(shape=rnn_cell.state_size)]
                x, hidden = rnn_cell(x, hidden_inputs)  # assume hidden is list of length 1 with tensor
                all_hidden_inputs.extend(hidden_inputs)
                all_hidden_outputs.extend(hidden)
            else:
                # multiple hiddens
                hidden_inputs = [tf.keras.Input(shape=size) for size in rnn_cell.state_size]
                x, hidden_outputs = rnn_cell(x, hidden_inputs)
                all_hidden_inputs.extend(hidden_inputs)
                all_hidden_outputs.extend(hidden_outputs)

        else:
            x = keras.layers.RNN(rnn_cell,
                                 batch_input_shape=(batch_size, seq_len,
                                                    x.shape[-1]),
                                 return_sequences=True,
                                 stateful=rnn_stateful,
                                 time_major=False)(x)

    x = keras.layers.Dense(units=4, activation='linear')(x)
    if single_step:
        ctrnn_model = keras.Model([inputs_image, *all_hidden_inputs], [x, *all_hidden_outputs])
    else:
        ctrnn_model = keras.Model([inputs_image], [x])

    return ctrnn_model

In [11]:
import copy
IMAGE_SHAPE = (144, 256, 3)

# helper classes that contain all the parameters in the generate_*_model functions
@dataclass
class ModelParams:
    # dataclasses can't have non-default follow default
    seq_len: int = field(default=False, init=True)
    image_shape: Tuple[int, int, int] = IMAGE_SHAPE
    augmentation_params: Optional[Dict] = None
    batch_size: Optional[int] = None
    single_step: bool = False
    no_norm_layer: bool = False


@dataclass
class NCPParams(ModelParams):
    seed: int = DEFAULT_NCP_SEED
        
@dataclass
class LSTMParams(ModelParams):
    rnn_sizes: List[int] = field(default=False, init=True)
    dropout: float = 0.1
    recurrent_dropout: float = 0.1
    rnn_stateful: bool = False


@dataclass
class CTRNNParams(ModelParams):
    rnn_sizes: List[int] = field(default=False, init=True)
    ct_network_type: str = 'ctrnn'
    config: Dict = field(default_factory=lambda: copy.deepcopy(DEFAULT_CFC_CONFIG))
    rnn_stateful: bool = False
    wiredcfc_seed: int = DEFAULT_NCP_SEED
        
def get_skeleton(params: ModelParams):
    """
    Returns a new model with randomized weights according to the parameters in params
    """
    if isinstance(params, NCPParams) or "NCPParams" in params.__class__.__name__:
        model_skeleton = generate_ncp_model(**asdict(params))
    elif isinstance(params, CTRNNParams) or "CTRNNParams" in params.__class__.__name__:
        model_skeleton = generate_ctrnn_model(**asdict(params))
    elif isinstance(params, LSTMParams) or "LSTMParams" in params.__class__.__name__:
        model_skeleton = generate_lstm_model(**asdict(params))
    else:
        raise ValueError(f"Could not parse param type {params.__class__}")
    return model_skeleton

def get_readable_name(params: Union[ModelParams, str]):
    """
    Extracts the model name from the class of params
    """
    if isinstance(params, str):
        params = eval_model_params(params)
    class_name = str(params.__class__.__name__)
    name = class_name.replace("Params", "").lower()
    return name

In [12]:
def get_output_normalization(root):
    training_output_mean_fn = os.path.join(root, 'stats', 'training_output_means.csv')
    if os.path.exists(training_output_mean_fn):
        print('Loading training data output means from: %s' % training_output_mean_fn)
        output_means = np.genfromtxt(training_output_mean_fn, delimiter=',')
    else:
        output_means = np.zeros(4)

    training_output_std_fn = os.path.join(root, 'stats', 'training_output_stds.csv')
    if os.path.exists(training_output_std_fn):
        print('Loading training data output std from: %s' % training_output_std_fn)
        output_stds = np.genfromtxt(training_output_std_fn, delimiter=',')
    else:
        output_stds = np.ones(4)

    return output_means, output_stds

def load_dataset_multi(root, image_size, seq_len, shift, stride, label_scale):
    file_ending = 'png'

    def sub_to_batch(sub_feature, sub_label):
        sib = sub_feature['input_image'].batch(seq_len, drop_remainder=True)
        slb = sub_label.batch(seq_len, drop_remainder=True)
        return tf.data.Dataset.zip(({"input_image":sib}, slb))
        # return sub.batch(seq_len, drop_remainder=True)

    dirs = sorted(os.listdir(root))
    dirs = [d for d in dirs if 'cached' not in d and 'stats' not in d and 'DS_Store' not in d]
    datasets = []
    print (dirs)
    output_means, output_stds = get_output_normalization(root)

    for (run_number, d) in tqdm(enumerate(dirs)):
        labels = np.genfromtxt(os.path.join(root, d, 'data_out.csv'), delimiter=',', skip_header=1, dtype=np.float32)
        print(os.path.join(root, d, 'data_out.csv'))

        if labels.shape[1] == 4:
            labels = (labels - output_means) / output_stds
            # labels = labels * label_scale
        elif labels.shape[1] == 5:
            labels = (labels[:, 1:] - output_means) / output_stds
            # labels = labels[:,1:] * label_scale
        else:
            raise Exception('Wrong size of input data (expected 4, got %d' % labels.shape[1])
        labels_dataset = tf.data.Dataset.from_tensor_slices(labels)

        n_images = len([fn for fn in os.listdir(os.path.join(root, d)) if file_ending in fn])
        dataset_np = np.empty((n_images, *image_size), dtype=np.uint8)
        print(n_images)

        for ix in range(n_images):
            # dataset_np[ix] = imread(os.path.join(root, d, '%06d.jpeg' % ix))
            # open image with rgb channels
            #img = Image.open(os.path.join(root, d, '%06d.%s' % (ix, file_ending)))
            img = Image.open(os.path.join(root, d, '%06d.%s' % (ix, file_ending))).convert('RGB')
            dataset_np[ix] = img

        images_dataset = tf.data.Dataset.from_tensor_slices(dataset_np)
        dataset = tf.data.Dataset.zip(({"input_image":images_dataset}, labels_dataset))
        dataset = dataset.window(seq_len, shift=shift, stride=stride, drop_remainder=True).flat_map(sub_to_batch)
        datasets.append(dataset)

    return datasets


def get_dataset_multi(root, image_size, seq_len, shift, stride, validation_ratio, label_scale, extra_data_root=None):
    ds = load_dataset_multi(root, image_size, seq_len, shift, stride, label_scale)
    print('n bags: %d' % len(ds))
    cnt = 0
    print(ds[0])

    for d in ds:
        for (ix, _) in enumerate(d):
            pass
            cnt += ix
    print('n windows: %d' % cnt)

    if extra_data_root is not None:
        ds_extra = load_dataset_multi(extra_data_root, image_size, seq_len, shift, stride, label_scale)
        print('\n\n Loaded Extra Dataset! \n\n')
        print('n extra bags: %d' % len(ds_extra))
        cnt = 0
        for d in ds_extra:
            for (ix, _) in enumerate(d):
                pass
            cnt += ix
        print('n extra windows: %d' % cnt)

    # indices = np.arange(len(ds))

    # The RNG used to split the validation data is deterministic here to prevent leakage between validation and training between runs
    # rng_val_split = np.random.default_rng(123)
    # rng_val_split.shuffle(indices)

    val_ix = int(len(ds) * validation_ratio)
    print('\nval_ix: %d\n' % val_ix)
    validation_datasets = ds[:val_ix]

    if extra_data_root is not None:
        training_datasets = ds[val_ix:] + ds_extra
        print('Total data length: %d' % len(training_datasets))
    else:
        training_datasets = ds[val_ix:]

    # if either dataset has length 0, trying to call flat map raises error that return type is wrong
    assert len(training_datasets) > 0 and len(validation_datasets) > 0, f"Training or validation dataset has no points!" \
                                                                        f"Train dataset len: {len(training_datasets)}" \
                                                                        f"Val dataset len: {len(validation_datasets)}"
    training = tf.data.Dataset.from_tensor_slices(training_datasets).flat_map(lambda x: x)
    validation = tf.data.Dataset.from_tensor_slices(validation_datasets).flat_map(lambda x: x)

    return training, validation

In [13]:
import functools
import os
from pathlib import Path
from typing import List, Dict, Any

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import argparse
import time

def tlen(dataset):
    for (ix, _) in enumerate(dataset):
        pass
    return ix


@tf.function
def sequence_augmentation(x, y, aug_params: Dict[str, Any]):
    """
    Apply augmentations where params have to be fixed per-sequence, and not per-sample. Therefore, these augmentations
    can't go in a layer, as TimeDistribtued would call the layer again and again for each timestep

    Note: to set breakpoints in a tf.function, you need to run tf.config.run_functions_eagerly(True) after import

    :param x: data input, has shape batch x seq_len x height x width x channels
    :param y: data labels, have shape batch x seq_len x 4
    :param aug_params: dictionary containing intensity of augmentations. Keys can include brightness, contrast, and
    saturation
    :return: augmented data input, same data labels
    """
    xi = x["input_image"]
    bright_range = aug_params.get("brightness", None)
    if bright_range is not None:
        delta = tf.random.uniform((), -bright_range, bright_range)
        xi = tf.image.adjust_brightness(xi, delta)

    contrast_range = aug_params.get("contrast", None)
    if contrast_range is not None:
        contrast_factor = tf.random.uniform((), 1 - contrast_range, 1 + contrast_range)
        xi = tf.image.adjust_contrast(xi, contrast_factor)

    saturation_range = aug_params.get("saturation", None)
    if saturation_range is not None:
        saturation_factor = tf.random.uniform((), 1 - saturation_range, 1 + saturation_range)
        xi = tf.image.adjust_saturation(xi, saturation_factor)

    return {"input_image":xi}, y


def train_model(model_params: ModelParams, data_dir: str = "./data", cached_data_dir: str = None,
                extra_data_dir: str = None, save_dir: str = "./model_checkpoints", batch_size: int = 32,
                epochs: int = 30, val_split: float = 0.1, hotstart: str = None, lr: float = 0.001, momentum: float = 0,
                opt: str = "adam", label_scale: float = 1, data_shift: int = 1, data_stride: int = 1,
                decay_rate: float = 0.95, callbacks: List = None, save_period: int = 1):
    # create model checkpoint directory if doesn't exist
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    # make sure data loading happens on CPU
    with tf.device('/cpu:0'):
        if cached_data_dir is not None:
            Path(cached_data_dir).mkdir(parents=True, exist_ok=True)
            data_folder = os.path.basename(data_dir)
            extra_data_str = f"_{os.path.basename(extra_data_dir)}" if extra_data_dir is not None else ""
            cached_training_fn = os.path.join(cached_data_dir, 'cached_dataset_%s%s_%d_%d_%d.tf' % (
                data_folder, extra_data_str, model_params.seq_len, data_stride, data_shift))
            cached_validation_fn = os.path.join(cached_data_dir, 'cached_dataset_%s%s_validation_%d_%d_%d.tf' % (
                data_folder, extra_data_str, model_params.seq_len, data_stride, data_shift))
            dataset_spec = os.path.join(cached_data_dir,
                                        f"cached_{data_folder}{extra_data_str}_{model_params.seq_len}_{data_stride}_{data_shift}_spec.txt")

        if cached_data_dir is not None and os.path.exists(cached_training_fn) and os.path.exists(
                cached_validation_fn) and os.path.exists(dataset_spec):
            # loading datasets in older versions of tensorflow requires a TensorSpec to describe
            with open(dataset_spec, "r") as f:
                spec_str = f.readlines()[0]
            spec: TensorSpec = eval(spec_str)
            print('Loading cached dataset from %s' % cached_training_fn)
            training_dataset = tf.data.experimental.load(cached_training_fn, spec)
            print('Loading cached dataset from %s' % cached_validation_fn)
            validation_dataset = tf.data.experimental.load(cached_validation_fn, spec)
        else:
            print('Loading data from: ' + data_dir)
            training_dataset, validation_dataset = get_dataset_multi(data_dir, IMAGE_SHAPE, model_params.seq_len,
                                                                     data_shift,
                                                                     data_stride, val_split, label_scale,
                                                                     extra_data_dir)

            if cached_data_dir is not None:
                print('Saving cached training data at %s' % cached_training_fn)
                tf.data.experimental.save(training_dataset, cached_training_fn)
                print('Saving cached validation data at %s' % cached_validation_fn)
                tf.data.experimental.save(validation_dataset, cached_validation_fn)
                with open(dataset_spec, "w") as f:
                    f.write(repr(training_dataset.element_spec))

        print('\n\nTraining Dataset Size: %d\n\n' % tlen(training_dataset))
        training_dataset = training_dataset.shuffle(100).batch(batch_size)
        # handle sequence augmentations differently
        seq_params = model_params.augmentation_params.get("sequence_params", None)
        if isinstance(seq_params, dict):
            print("Performing sequence aug on training dataset")
            seq_aug_fn = functools.partial(sequence_augmentation, aug_params=seq_params)
            training_dataset = training_dataset.map(
                seq_aug_fn, num_parallel_calls=tf.data.AUTOTUNE
            )
        validation_dataset = validation_dataset.batch(batch_size)
        # remove annoying TF warning about dataset sharding across multiple GPUs
        options = tf.data.Options()
        options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
        training_dataset = training_dataset.with_options(options)
        validation_dataset = validation_dataset.with_options(options)
        # Have GPU prefetch next training batch while first one runs
        training_dataset = training_dataset.prefetch(tf.data.AUTOTUNE)
        validation_dataset = validation_dataset.prefetch(tf.data.AUTOTUNE)

    lr_schedule = keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=lr, decay_steps=500,
                                                              decay_rate=decay_rate, staircase=True)

    if opt == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=lr_schedule)
    elif opt == 'sgd':
        optimizer = keras.optimizers.SGD(learning_rate=lr_schedule, momentum=momentum)
    else:
        raise Exception('Unsupported optimizer type %s' % opt)

    time_str = time.strftime("%Y:%m:%d:%H:%M:%S")

    file_path = os.path.join(save_dir, 'model-%s_seq-%d_lr-%f_epoch-{epoch:03d}'
                                       '_val-loss:{val_loss:.4f}_train-loss:{loss:.4f}_mse:{mse:.4f}_%s.hdf5' % (
                                 get_readable_name(model_params), model_params.seq_len, lr, time_str))

    checkpoint_callback = keras.callbacks.ModelCheckpoint(filepath=file_path, save_weights_only=True,
                                                          save_best_only=False, save_freq='epoch', period=save_period)

    if callbacks is None:
        callbacks = []

    callbacks.append(checkpoint_callback)
    print(f"Saving checkpoints at {file_path}")

    # use data parallelism to split data across GPUs
    gpus = tf.config.list_logical_devices('GPU')
    strategy = tf.distribute.MirroredStrategy(gpus)
    with strategy.scope():
        model = get_skeleton(params=model_params)
        model.compile(optimizer=optimizer, loss="mean_squared_error", metrics=['mse'])
        # Load pretrained weights
        if hotstart is not None:
            model.load_weights(hotstart)

        model.summary(line_length=80)

    # Train
    history = model.fit(x=training_dataset, validation_data=validation_dataset, epochs=epochs,
                        use_multiprocessing=False, workers=1, max_queue_size=5, verbose=1, callbacks=callbacks)
    return history, time_str

In [16]:
model = input('The type of model (ncp, lstm, ctrnn)') or 'ncp'
ct_type = input('The type of the continuous model (ctrnn, node, cfc, ctgru, grud, mmrnn, mixedcfc, bidirect, vanilla, phased, gruode, hawk, ltc)') or 'ctrnn'
rnn_sizes = list(map(int, input('Select the size of RNN network you would like to train').split()))
data_dir = input('Path to training data') or './data'
test_data_dir = input('Path to test data') or './data'
cached_data_dir = input('Path to pre-cached dataset') or None
extra_data_dir = input('Path to extra training data, used for training but not validation') or None
save_dir = input('Path to save checkpoints') or './model_checkpoints'
history_dir = input('Path to save history') or './histories'
batch_size = int(input('Number of sequences in one training batch') or 32) 
seq_len = int(input('Number of data points per sequence within each batch')or 64) 
epochs = int(input('Number of epochs to train for') or 30) 
val_split = float(input('Fraction of dataset that becomes validation set')or 0.1) 
hotstart = input('Starting weights to use for pretraining') or None
lr = float(input('Learning Rate"')or 0.001) 
momentum = float(input('Momentum (for use with SGD)')or 0.0) 
opt = input('Optimizer to use (adam, sgd)') or 'adam'
augmentation = bool(input('Whether to turn on data augmentation in network')or True) 
label_scale = float(input('Scale factor to apply to labels')or 1) 
translation_factor = float(input('Amount to (randomly) translate width and height (0 - 1.0).') or 0.1) 
rotation_factor = float(input('Amount to (randomly) rotate (0.0 - 1.0).') or 0.1) 
zoom_factor = float(input('Amount to (randomly) zoom.')or 0.1) 
data_stride = int(input('Stride within image sequence. Default=1.') or 1) 
data_shift = int(input('Window shift between windows. Default=1.')or 1) 
decay_rate = float(input("Exponential decay rate of the lr scheduler")or 0.95) 
ncp_seed = int(input("Seed for ncp") or 2222)

augmentation_params = {"translation_factor": translation_factor, "rotation_factor": rotation_factor,
                           "zoom_factor": zoom_factor} if augmentation else None

if model == "ncp":
        model_params_constructed = NCPParams(seq_len=seq_len,
                                             augmentation_params=augmentation_params, seed=ncp_seed)
elif model == "lstm":
        model_params_constructed = LSTMParams(seq_len=seq_len,
                                              augmentation_params=augmentation_params, rnn_sizes=rnn_sizes, )
elif model == "ctrnn":
        model_params_constructed = CTRNNParams(seq_len=seq_len,
                                               augmentation_params=augmentation_params, rnn_sizes=rnn_sizes,
                                               ct_network_type=ct_type)
else:
        raise ValueError(f"Passed in illegal model type {model}")


train_model(data_dir=data_dir, epochs=epochs, val_split=val_split,
                opt=opt, lr=lr, data_shift=data_shift, data_stride=data_stride,
                batch_size=batch_size, save_dir=save_dir, hotstart=hotstart, momentum=momentum,
                cached_data_dir=cached_data_dir, label_scale=label_scale,
                model_params=model_params_constructed, decay_rate=decay_rate)

# {
#   "model-ctrnn_ctrnn_seq-64_lr-0.000292_epoch-096_val-loss:0.0706_train-loss:0.0050_mse:0.0050_2022:04:18:23:58:35.hdf5": "CTRNNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[252], ct_network_type='ctrnn', config={'clipnorm': 1, 'backbone_activation': 'silu', 'backbone_dr': 0.1, 'forget_bias': 1.6, 'backbone_units': 128, 'backbone_layers': 1, 'weight_decay': 1e-06}, rnn_stateful=False, wiredcfc_seed=22222)",
#   "model-ctrnn_gruode_seq-64_lr-0.000925_epoch-099_val-loss:0.0758_train-loss:0.0031_mse:0.0031_2022:04:16:20:47:58.hdf5": "CTRNNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[198], ct_network_type='gruode', config={'clipnorm': 1, 'backbone_activation': 'silu', 'backbone_dr': 0.1, 'forget_bias': 1.6, 'backbone_units': 128, 'backbone_layers': 1, 'weight_decay': 1e-06}, rnn_stateful=False, wiredcfc_seed=22222)",
#   "model-ctrnn_cfc_seq-64_lr-0.000183_epoch-092_val-loss:0.0767_train-loss:0.0064_mse:0.0063_2022:04:15:21:21:24.hdf5": "CTRNNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[193], ct_network_type='cfc', config={'clipnorm': 1, 'backbone_activation': 'silu', 'backbone_dr': 0.1, 'forget_bias': 3.0092689425990775, 'backbone_units': 147, 'backbone_layers': 2, 'weight_decay': 4.239514500024343e-08}, rnn_stateful=False, wiredcfc_seed=22222)",
#   "model-ncp_seq-64_lr-0.000291_epoch-096_val-loss:0.0720_train-loss:0.0104_mse:0.0104_2022:04:15:05:11:37.hdf5": "NCPParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, seed=22224)",
#   "model-ctrnn_wiredcfccell_seq-64_lr-0.000479_epoch-100_val-loss:0.0855_train-loss:0.0037_mse:0.0037_2022:04:18:03:32:09.hdf5": "CTRNNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[196], ct_network_type='wiredcfccell', config={'clipnorm': 1, 'backbone_activation': 'silu', 'backbone_dr': 0.1, 'forget_bias': 1.6, 'backbone_units': 128, 'backbone_layers': 1, 'weight_decay': 1e-06}, rnn_stateful=False, wiredcfc_seed=22224)",
#   "model-tcn_seq-64_lr-0.000300_epoch-099_val-loss:0.0561_train-loss:0.0028_mse:0.0028_2022:04:18:23:58:35.hdf5": "TCNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, nb_filters=236, kernel_size=5, dilations=[1, 2, 4], dropout=0.25417245152621387)",
#   "model-ctrnn_ltc_seq-64_lr-0.000506_epoch-098_val-loss:0.0622_train-loss:0.0086_mse:0.0086_2022:04:15:21:21:24.hdf5": "CTRNNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[69], ct_network_type='ltc', config={'clipnorm': 1, 'backbone_activation': 'silu', 'backbone_dr': 0.1, 'forget_bias': 1.6, 'backbone_units': 128, 'backbone_layers': 1, 'weight_decay': 1e-06}, rnn_stateful=False, wiredcfc_seed=22222)",
#   "model-lstm_seq-64_lr-0.000290_epoch-085_val-loss:0.0711_train-loss:0.0023_mse:0.0023_2022:04:15:05:11:33.hdf5": "LSTMParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[193], dropout=0.15273092510550293, recurrent_dropout=0.15273092510550293, rnn_stateful=False)",
#   "model-ctrnn_mixedcfc_seq-64_lr-0.000087_epoch-092_val-loss:0.0626_train-loss:0.0074_mse:0.0073_2022:04:16:20:47:58.hdf5": "CTRNNParams(seq_len=64, image_shape=(144, 256, 3), augmentation_params={'noise': 0.05, 'sequence_params': {'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.4}}, batch_size=None, single_step=False, no_norm_layer=False, rnn_sizes=[115], ct_network_type='mixedcfc', config={'clipnorm': 1, 'backbone_activation': 'silu', 'backbone_dr': 0.1, 'forget_bias': 3.1633364953559684, 'backbone_units': 253, 'backbone_layers': 1, 'weight_decay': 9.356617874241273e-08}, rnn_stateful=False, wiredcfc_seed=22222)"
# }


The type of model (ncp, lstm, ctrnn) ctrnn
The type of the continuous model (ctrnn, node, cfc, ctgru, grud, mmrnn, mixedcfc, bidirect, vanilla, phased, gruode, hawk, ltc) cfc
Select the size of RNN network you would like to train 193
Path to training data /kaggle/input/data
Path to test data 
Path to pre-cached dataset 
Path to extra training data, used for training but not validation 
Path to save checkpoints /kaggle/working/model_checkpoints
Path to save history /kaggle/working/history
Number of sequences in one training batch 
Number of data points per sequence within each batch 64
Number of epochs to train for 100
Fraction of dataset that becomes validation set 
Starting weights to use for pretraining 
Learning Rate" 0.000183
Momentum (for use with SGD) 
Optimizer to use (adam, sgd) adam
Whether to turn on data augmentation in network 
Scale factor to apply to labels 
Amount to (randomly) translate width and height (0 - 1.0). 
Amount to (randomly) rotate (0.0 - 1.0). 
Amount to (ra

Loading data from: /kaggle/input/data
['1628106140.64', '1628106314.97', '1628106438.69', '1628106866.84', '1628107077.97', '1628107179.39', '1628107388.53', '1628107517.78', '1628107620.45', '1628107723.67']


0it [00:00, ?it/s]

/kaggle/input/data/1628106140.64/data_out.csv
2995


1it [00:25, 25.71s/it]

...

9it [03:12, 20.40s/it]

/kaggle/input/data/1628107723.67/data_out.csv
2320


10it [03:32, 21.21s/it]


n bags: 10
<_FlatMapDataset element_spec=({'input_image': TensorSpec(shape=(64, 144, 256, 3), dtype=tf.uint8, name=None)}, TensorSpec(shape=(64, 4), dtype=tf.float64, name=None))>
n windows: 117449

val_ix: 1



Training Dataset Size: 1346


Saving checkpoints at /kaggle/working/model_checkpoints/model-ctrnn_seq-64_lr-0.000183_epoch-{epoch:03d}_val-loss:{val_loss:.4f}_train-loss:{loss:.4f}_mse:{mse:.4f}_2023:11:24:01:47:14.hdf5
(None, 64, 128) get_network_trunk
(None, 64, 128)
Model: "model"
________________________________________________________________________________
 Layer (type)                       Output Shape                    Param #     
 input_image (InputLayer)           [(None, 64, 144, 256, 3)]       0           
                                                                                
 rescaling (Rescaling)              (None, 64, 144, 256, 3)         0           
                                                                                
 time_distributed

(<keras.src.callbacks.History at 0x7fe72db17130>, '2023:11:24:01:47:14')

In [ ]:
os.listdir('/kaggle/input/data')